# Vision AI describer


In [ ]:
!pip install anthropic gradio --quiet

In [ ]:
!pip install transformers gradio torch torchvision --quiet

# Last version


In [ ]:
# ============================================================
# 🖼️ Vision AI — Image Describer (All-in-One)
# ============================================================

# ─── CELL 1: Install ─────────────────────────────────────────
# !pip install transformers gradio torch torchvision --quiet

# ─── CELL 2: Load model ──────────────────────────────────────
import os
os.environ["GRADIO_DEFAULT_LOCALE"] = "en"   # 👈 Force English UI

from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch
import gradio as gr

print("⏳ Loading model...")

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-large"
).to(device)

print(f"✅ Model loaded! Running on: {device.upper()}")


# ─── CELL 3: Core ask function ───────────────────────────────

def ask(pil_image, prompt=None, max_tokens=80, beams=7):
    """Run a single prompt through BLIP and return clean result."""
    if prompt:
        inputs = processor(pil_image, prompt, return_tensors="pt").to(device)
    else:
        inputs = processor(pil_image, return_tensors="pt").to(device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        num_beams=beams,
        min_length=10,
        repetition_penalty=1.5,
        length_penalty=1.3,
        early_stopping=True,
    )
    result = processor.decode(out[0], skip_special_tokens=True).strip()

    # Remove echoed prompt prefix
    if prompt and result.lower().startswith(prompt.lower()):
        result = result[len(prompt):].strip()

    # Capitalize and add period
    if result:
        result = result[0].upper() + result[1:]
        if not result.endswith((".", "!", "?")):
            result += "."
    return result


# ─── CELL 4: GPT-style paragraph builder ─────────────────────

def build_gpt_description(pil_image):
    """Run 15 layered questions and weave into 4 paragraphs."""

    base       = ask(pil_image)
    subject    = ask(pil_image, "the main subject of this image is")
    setting    = ask(pil_image, "the setting or location of this image is")
    time       = ask(pil_image, "the time of day or lighting condition is")

    foreground = ask(pil_image, "in the foreground of this image")
    background = ask(pil_image, "in the background of this image")
    colors     = ask(pil_image, "the dominant colors in this image are")
    lighting   = ask(pil_image, "the lighting in this image is")
    textures   = ask(pil_image, "the textures and materials visible are")

    people     = ask(pil_image, "the people or figures in this image are")
    objects    = ask(pil_image, "the key objects visible in this image are")
    actions    = ask(pil_image, "the action or activity taking place is")

    mood       = ask(pil_image, "the mood and atmosphere of this image is")
    emotion    = ask(pil_image, "this image makes the viewer feel")
    style      = ask(pil_image, "the visual style or genre of this image is")

    para1 = (
        f"{base} "
        f"The main subject is {subject.lower().rstrip('.')}. "
        f"The scene is set {setting.lower().rstrip('.')}, "
        f"with {time.lower().rstrip('.')}."
    )
    para2 = (
        f"Looking at the composition, {foreground.lower().rstrip('.')} "
        f"while {background.lower().rstrip('.')}. "
        f"The dominant colors are {colors.lower().rstrip('.')}, "
        f"and the lighting is {lighting.lower().rstrip('.')}. "
        f"Notable textures include {textures.lower().rstrip('.')}."
    )
    para3 = (
        f"The image features {people.lower().rstrip('.')}. "
        f"Key objects include {objects.lower().rstrip('.')}. "
        f"The activity depicted is {actions.lower().rstrip('.')}."
    )
    para4 = (
        f"In terms of atmosphere, {mood.lower().rstrip('.')}. "
        f"{emotion} "
        f"The visual style can be described as {style.lower().rstrip('.')}."
    )

    return f"{para1}\n\n{para2}\n\n{para3}\n\n{para4}"


# ─── CELL 5: Multi-prompt modes ──────────────────────────────

MODE_PROMPTS = {
    "📝 Full Description": [
        None,
        "a detailed description of this image shows",
        "in the foreground of this image",
        "in the background of this image",
    ],
    "🎨 Colors & Composition": [
        "the dominant colors in this image are",
        "the lighting in this image is",
        "the composition and layout of this image shows",
        "the visual style of this image is",
    ],
    "👶 Child-Friendly Check": [
        "this image is appropriate for children because",
        "the content of this image",
        "the theme of this image is",
    ],
    "📦 List Objects": [
        "the main objects in this image are",
        "in the foreground there is",
        "in the background there is",
        "the people or figures in this image are",
    ],
    "😊 Mood & Emotion": [
        "the mood of this image is",
        "the atmosphere in this image feels",
        "the emotion conveyed by this image is",
        "this image makes the viewer feel",
    ],
    "🌍 Scene & Setting": [
        "the setting of this image is",
        "the location shown in this image is",
        "the time of day in this image appears to be",
        "the environment in this image is",
    ],
    "🔍 Fine Details": [
        "a close look at this image reveals",
        "the textures in this image include",
        "notable details in this image are",
        "the small details visible in this image are",
    ],
}


def run_multi_modes(pil_image, selected_modes):
    """Run selected multi-prompt modes and return combined output."""
    all_sections = []

    for mode in selected_modes:
        prompts = MODE_PROMPTS[mode]
        lines = []
        for prompt in prompts:
            result = ask(pil_image, prompt)
            if result and len(result) > 5:
                label = prompt.capitalize() if prompt else "General"
                lines.append(f"  • {label}:\n    {result}")
        if lines:
            section = f"{'─'*40}\n{mode}\n{'─'*40}\n" + "\n".join(lines)
            all_sections.append(section)

    return "\n\n".join(all_sections)


# ─── CELL 6: Main function ───────────────────────────────────

def describe_image(image, use_gpt_mode, selected_modes):
    if image is None:
        return "⚠️ Please upload an image."

    pil_image = Image.fromarray(image).convert("RGB")
    results = []

    try:
        # GPT-style comprehensive paragraph
        if use_gpt_mode:
            gpt_result = build_gpt_description(pil_image)
            results.append(
                f"{'═'*40}\n🧠 COMPREHENSIVE DESCRIPTION\n{'═'*40}\n{gpt_result}"
            )

        # Multi-select modes
        if selected_modes:
            multi_result = run_multi_modes(pil_image, selected_modes)
            if multi_result:
                results.append(multi_result)

        if not results:
            return "⚠️ Please enable Comprehensive Mode or select at least one mode below."

        return "\n\n".join(results)

    except Exception as e:
        return f"❌ Error: {str(e)}"


# ─── CELL 7: Launch Gradio app ───────────────────────────────

with gr.Blocks(title="Vision AI — All-in-One", theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🖼️ Vision AI — Image Describer")


    with gr.Row():

        # ── Left column ──────────────────────────────────────
        with gr.Column(scale=1):
            image_input = gr.Image(
                type="numpy",
                label="Upload Image",
                height=300
            )

            gr.Markdown("### 🧠 Comprehensive Mode")
            gpt_toggle = gr.Checkbox(
                value=True,
                label="✦ GPT-Style Paragraph Description (15 layered questions → 4 paragraphs)"
            )

            gr.Markdown("### 🗂️ Additional Modes (multi-select)")
            mode_input = gr.CheckboxGroup(
                choices=list(MODE_PROMPTS.keys()),
                value=[],
                label="Select one or more extra modes"
            )

            with gr.Row():
                select_all_btn = gr.Button("✅ Select All", size="sm")
                clear_btn      = gr.Button("🗑️ Clear All",  size="sm")

            analyze_btn = gr.Button("✦ Analyze Image", variant="primary", size="lg")

            gr.Markdown(
                "> ⏱️ Comprehensive mode takes **20–40s on CPU**, "
                "**5–10s on GPU**. Enable T4 GPU in Colab for best speed."
            )

        # ── Right column ─────────────────────────────────────
        with gr.Column(scale=1):
            output_text = gr.Textbox(
                label="AI Description",
                lines=28,
                show_copy_button=True
            )

    select_all_btn.click(fn=lambda: list(MODE_PROMPTS.keys()), inputs=[], outputs=mode_input)
    clear_btn.click(fn=lambda: [], inputs=[], outputs=mode_input)

    analyze_btn.click(
        fn=describe_image,
        inputs=[image_input, gpt_toggle, mode_input],
        outputs=output_text
    )

demo.launch(share=True, debug=False, server_name="0.0.0.0")

⏳ Loading model...


Loading weights:   0%|          | 0/616 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-large
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded! Running on: CPU


/tmp/ipykernel_766/2600944460.py:212: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Vision AI — All-in-One", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b3b1916e89220cfb32.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
